<a href="https://colab.research.google.com/github/VinishReddyK/zipcast-epub/blob/main/notebooks/colab_zipcast_qwen3tts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# zipcast: epub → audiobook (Qwen3-TTS)

This notebook runs a small web GUI for the whole conversion -- upload epubs,
pick chapters, tune chunk/batch size, preview a described voice, and watch
live progress -- instead of editing cells by hand.

**Steps to use this notebook:**
1. Set **Runtime > Change runtime type > T4 GPU**.
2. Run every cell top to bottom.
3. The last cell prints a `https://*.trycloudflare.com` link -- open it.
4. In the page: upload your `.epub`(s), set chapters/chunk size/batch size/voice,
   click **Start conversion**, and watch the live progress bar + ETA. Finished
   `.m4b` files get a download link as each book completes.


In [ ]:
import os

# reduce CUDA memory fragmentation on long multi-chapter runs (must be set
# before torch initializes a CUDA context)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

REPO_URL = "https://github.com/VinishReddyK/zipcast-epub.git"  # @param {type:"string"}
REPO_DIR = "/content/zipcast"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


In [ ]:
# install ffmpeg (for m4b muxing) and the epub-parsing + Qwen3-TTS + web GUI dependencies
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip install -q -r {REPO_DIR}/colab/requirements.txt


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. go to Runtime > Change runtime type > T4 GPU, "
          "otherwise synthesis will be very slow.")


## Launch the GUI

Starts the local web app, then opens a free Cloudflare "quick tunnel" to it --
no account, no signup, no auth token. The tunnel prints a temporary
`https://*.trycloudflare.com` URL below; open it in a new tab.

This cell then keeps running and mirrors every conversion job's logs and
progress bars (overall + current chapter) right here in the notebook too --
start a conversion from the GUI page, then watch this cell's output.


In [ ]:
import os
import re
import subprocess
import threading
import time

CLOUDFLARED_PATH = "/content/cloudflared"
if not os.path.exists(CLOUDFLARED_PATH):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {CLOUDFLARED_PATH}
    !chmod +x {CLOUDFLARED_PATH}

from colab.webapp import start_server_in_background

start_server_in_background(port=5000)
time.sleep(2)  # give Flask a moment to bind before the tunnel connects to it

proc = subprocess.Popen(
    [CLOUDFLARED_PATH, "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

url_found = threading.Event()
tunnel_url = {"value": None}

def _watch():
    for line in proc.stdout:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match and not url_found.is_set():
            tunnel_url["value"] = match.group(0)
            url_found.set()

threading.Thread(target=_watch, daemon=True).start()
url_found.wait(timeout=30)

if tunnel_url["value"]:
    print(f"\nopen the GUI here: {tunnel_url['value']}\n")
else:
    print("tunnel did not report a URL within 30s -- check the cell output above for errors.")

# mirror every job's progress + logs into this cell too (in addition to the
# GUI), since prints from the background server thread above aren't reliably
# visible in any cell once this one finishes -- so we actively poll the same
# progress stream the browser uses, in a loop, for as long as this cell runs.
import json as _json
from tqdm.auto import tqdm
import requests

API = "http://localhost:5000/api"
print("watching for conversion jobs started from the GUI (run a job there now)...\n")

while True:
    job_id = None
    while job_id is None:
        job_id = requests.get(f"{API}/jobs/active").json().get("job_id")
        if job_id is None:
            time.sleep(1)

    print(f"=== mirroring job {job_id} ===")
    overall_bar, chapter_bar = None, None
    with requests.get(f"{API}/jobs/{job_id}/events", stream=True) as resp:
        for raw_line in resp.iter_lines(decode_unicode=True):
            if not raw_line or not raw_line.startswith("data:"):
                continue
            event = _json.loads(raw_line[len("data:"):].strip())
            kind = event.get("event")

            if kind == "log":
                print(event["message"])
            elif kind == "plan_ready":
                overall_bar = tqdm(total=event["chunks_total"], unit="chunk", desc="overall", position=0)
            elif kind == "book_start":
                print(f"\n=== book {event['book_index']}/{event['books_total']}: {event['book']} ({event['chapters_total']} chapters) ===")
            elif kind == "chapter_start":
                if chapter_bar is not None:
                    chapter_bar.close()
                chapter_bar = tqdm(
                    total=event["chunks_in_chapter"], unit="chunk",
                    desc=event["chapter_title"][:30], position=1, leave=False,
                )
            elif kind == "chapter_skip":
                print(f"  skip (already synthesized): {event['chapter_title']}")
            elif kind == "chunk_progress":
                if overall_bar is not None:
                    overall_bar.n = event["chunks_done"]
                    eta = event["eta_sec"]
                    overall_bar.set_postfix(rate=f"{event['chunks_per_sec']:.2f}/s", eta=f"{eta:.0f}s" if eta is not None else "?")
                    overall_bar.refresh()
                if chapter_bar is not None:
                    chapter_bar.n = event["chunks_done_chapter"]
                    chapter_bar.refresh()
            elif kind == "chapter_done":
                if chapter_bar is not None:
                    chapter_bar.close()
                    chapter_bar = None
                print(f"  ✓ {event['chapter_title']}")
            elif kind == "book_done":
                print(f"✓✓ finished: {event['output_path']}")
            elif kind == "error":
                print(f"ERROR: {event['message']}")
            elif kind == "all_done":
                if overall_bar is not None:
                    overall_bar.close()
                print(f"\nall done: {len(event['outputs'])} audiobook(s) generated")
                for p in event["outputs"]:
                    print(f"  {p}")
            elif kind == "stream_end":
                break

    print("\nwaiting for the next job...\n")
